# 02 — Canonical time-series EDA

This notebook explores canonical telemetry before feature engineering or
modelling. It runs unchanged for Telecom and Petrobras 3W and reads only
`SPEC-CORE`.

It does not read labels, fill missing values, remove outliers, create anomaly
scores, or assume that any interval is normal.


## 1. Setup

Change `SECTOR` to select a completed canonical run. Locally, inputs and
outputs default to `~/anomaly_detection_data/`; in Colab they default to
`MyDrive/anomaly_detection/`. Population summaries use
all telemetry through DuckDB. Heavier plots use a few typical episodes.


In [ ]:
import hashlib
import os
import shutil
import sys
import tempfile
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss

try:
    import duckdb
except ImportError:
    !pip install -q duckdb
    import duckdb

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

default_data_root = (
    Path("/content/drive/MyDrive/anomaly_detection")
    if IN_COLAB else Path.home() / "anomaly_detection_data"
)
data_root_setting = (
    os.getenv("ANOMALY_DATA_ROOT")
    or os.getenv("ANOMALY_DRIVE_ROOT")  # legacy Colab name
)
DATA_ROOT = Path(data_root_setting or default_data_root).expanduser()

if IN_COLAB:
    default_notebook_home = DATA_ROOT / "research" / "milestone1"
elif (Path.cwd() / "milestone1_core.py").is_file():
    default_notebook_home = Path.cwd()
else:
    default_notebook_home = Path.cwd() / "notebooks" / "drive_research"
NOTEBOOK_HOME = Path(
    os.getenv("ANOMALY_NOTEBOOK_HOME", default_notebook_home)
).expanduser()
if not (NOTEBOOK_HOME / "milestone1_core.py").is_file():
    raise FileNotFoundError(
        f"milestone1_core.py was not found in {NOTEBOOK_HOME}"
    )
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from milestone1_core import CORE_VERSION, new_output_directory, read_json, write_json

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

EDA_VERSION = "0.5.0"
SECTOR = os.getenv("EDA_SECTOR", "telecom")  # "telecom" or "petrobras_3w"
CANONICAL_RUN_IDS = {
    "telecom": "telecom_core_v0_9_1_run1",
    "petrobras_3w": "petrobras_3w_core_v0_9_1_run1",
}
if SECTOR not in CANONICAL_RUN_IDS:
    raise ValueError(f"Choose one of {list(CANONICAL_RUN_IDS)}")

CANONICAL_RUN_ID = os.getenv("EDA_CANONICAL_RUN_ID", CANONICAL_RUN_IDS[SECTOR])
EDA_RUN_ID = os.getenv(
    "EDA_RUN_ID", f"{SECTOR}_eda_v0_5_core_v0_9_1_run1"
)
CORE_ROOT = (
    DATA_ROOT / "outputs" / "canonical" / f"v{CORE_VERSION}"
    / SECTOR / CANONICAL_RUN_ID / "SPEC-CORE"
)
EDA_ROOT = DATA_ROOT / "outputs" / "eda" / f"v{EDA_VERSION}" / SECTOR / EDA_RUN_ID

ENTITY_IDS = tuple(filter(None, os.getenv("EDA_ENTITY_IDS", "").split(",")))
ENTITY_LIMIT = int(os.getenv("EDA_ENTITY_LIMIT", "10"))
MAX_ANALYSIS_ROWS = int(os.getenv("EDA_MAX_ANALYSIS_ROWS", "2000000"))
MAX_PLOT_METRICS = int(os.getenv("EDA_MAX_PLOT_METRICS", "8"))
MAX_STATE_PLOTS = int(os.getenv("EDA_MAX_STATE_PLOTS", "2"))
MIN_SEASONAL_CYCLES = 6
SAVE_OUTPUTS = os.getenv("SAVE_EDA_OUTPUTS", "1") == "1"

figure_temp = tempfile.TemporaryDirectory()
FIGURE_CACHE = Path(figure_temp.name)

def save_plot(fig, name):
    fig.tight_layout()
    fig.savefig(FIGURE_CACHE / name, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)

display(pd.Series({
    "runtime": "Colab + Drive" if IN_COLAB else "local Python",
    "data_root": str(DATA_ROOT),
    "code_root": str(NOTEBOOK_HOME),
    "sector": SECTOR,
    "spec_core": str(CORE_ROOT),
    "eda_output": str(EDA_ROOT),
    "entities": ENTITY_IDS or f"stable sample of {ENTITY_LIMIT}",
}, name="value").to_frame())


## 2. Inspect the canonical data

The catalogue explains metric names, measurement kinds, units, sampling modes
and expected cadence. Telemetry is displayed in canonical long form and a
small familiar wide view.


In [ ]:
if not CORE_ROOT.is_dir():
    raise FileNotFoundError(f"Run Notebook 01B first: {CORE_ROOT}")

manifest_path = CORE_ROOT / "manifest.json"
catalogue = pd.read_parquet(CORE_ROOT / "metric_catalogue.parquet")
registry = pd.read_parquet(CORE_ROOT / "entity_registry.parquet")
episodes = pd.read_parquet(CORE_ROOT / "observation_episodes.parquet")
collection_gaps = pd.read_parquet(CORE_ROOT / "collection_gaps.parquet")
telemetry_parts = sorted((CORE_ROOT / "telemetry").glob("part-*.parquet"))
if not telemetry_parts:
    raise FileNotFoundError("No canonical telemetry parts found")

core_manifest = read_json(manifest_path)
assert core_manifest["contract_version"] == CORE_VERSION
assert all(path.resolve().is_relative_to(CORE_ROOT.resolve()) for path in telemetry_parts)

telemetry_glob = str(CORE_ROOT / "telemetry" / "*.parquet").replace("'", "''")
connection = duckdb.connect()
connection.execute(f"CREATE VIEW telemetry AS SELECT * FROM read_parquet('{telemetry_glob}')")

sample = connection.execute('''
    SELECT * FROM telemetry
    ORDER BY entity_id, episode_id, event_ts, metric_id
    LIMIT 200
''').df()
sample_wide = sample.pivot_table(
    index=["event_ts", "entity_id", "episode_id"],
    columns="metric_id", values="value", aggfunc="first",
).reset_index()

display(catalogue)
display(registry.head(10))
display(episodes.head(10))
display(sample.head(20))
display(sample_wide.head(10))


## 3. Full-population structure and quality

An invalid row is an attempted observation with no usable value. Periodic
coverage measures absent expected rows. Robust 1st/99th percentiles and the
extreme-to-central span expose suspicious tails without inventing sector limits.
For 3W recordings, coverage between separate files is intentionally undefined.


In [ ]:
structural = connection.execute('''
    SELECT count(*) AS rows,
           count(DISTINCT entity_id) AS entities,
           count(DISTINCT episode_id) AS episodes,
           count(DISTINCT metric_id) AS metrics,
           min(event_ts) AS first_ts,
           max(event_ts) AS last_ts,
           count(*) - count(DISTINCT (event_ts, entity_id, episode_id, metric_id))
               AS duplicate_keys
    FROM telemetry
''').df()

value_summary = connection.execute('''
    SELECT metric_id,
           count(*) AS rows,
           avg(CASE WHEN quality_code <> 'invalid' AND value IS NOT NULL THEN 1.0 ELSE 0.0 END)
               AS valid_rate,
           avg(CASE WHEN quality_code = 'clipped' THEN 1.0 ELSE 0.0 END) AS clipped_rate,
           avg(CASE WHEN quality_code <> 'invalid' AND value IS NOT NULL
                    THEN CASE WHEN isfinite(value) THEN 1.0 ELSE 0.0 END END) AS finite_rate,
           approx_count_distinct(value) AS approximate_unique_values,
           min(CASE WHEN quality_code <> 'invalid' AND isfinite(value) THEN value END) AS minimum,
           approx_quantile(CASE WHEN quality_code <> 'invalid' AND isfinite(value) THEN value END, 0.01) AS q01,
           approx_quantile(CASE WHEN quality_code <> 'invalid' AND isfinite(value) THEN value END, 0.05) AS q05,
           approx_quantile(CASE WHEN quality_code <> 'invalid' AND isfinite(value) THEN value END, 0.50) AS median,
           approx_quantile(CASE WHEN quality_code <> 'invalid' AND isfinite(value) THEN value END, 0.95) AS q95,
           approx_quantile(CASE WHEN quality_code <> 'invalid' AND isfinite(value) THEN value END, 0.99) AS q99,
           max(CASE WHEN quality_code <> 'invalid' AND isfinite(value) THEN value END) AS maximum,
           avg(CASE WHEN quality_code <> 'invalid' AND value IS NOT NULL AND isfinite(value)
                    THEN CASE WHEN value = 0 THEN 1.0 ELSE 0.0 END END) AS zero_rate,
           avg(CASE WHEN quality_code <> 'invalid' AND isfinite(value) THEN value END) AS mean,
           var_samp(CASE WHEN quality_code <> 'invalid' AND isfinite(value) THEN value END) AS variance
    FROM telemetry
    GROUP BY metric_id
''').df()

series_summary = connection.execute('''
    SELECT entity_id, episode_id, metric_id,
           count(*) AS rows,
           avg(CASE WHEN quality_code <> 'invalid' AND value IS NOT NULL THEN 1.0 ELSE 0.0 END)
               AS valid_rate,
           avg(CASE WHEN quality_code = 'clipped' THEN 1.0 ELSE 0.0 END) AS clipped_rate,
           min(CASE WHEN quality_code <> 'invalid' AND isfinite(value) THEN value END) AS minimum,
           approx_quantile(CASE WHEN quality_code <> 'invalid' AND isfinite(value) THEN value END, 0.25) AS q25,
           approx_quantile(CASE WHEN quality_code <> 'invalid' AND isfinite(value) THEN value END, 0.50) AS series_median,
           approx_quantile(CASE WHEN quality_code <> 'invalid' AND isfinite(value) THEN value END, 0.75) AS q75,
           max(CASE WHEN quality_code <> 'invalid' AND isfinite(value) THEN value END) AS maximum
    FROM telemetry
    GROUP BY entity_id, episode_id, metric_id
''').df().merge(catalogue, on="metric_id", how="left", validate="many_to_one")

series_summary = series_summary.merge(
    episodes[["entity_id", "episode_id", "observed_from", "observed_to"]],
    on=["entity_id", "episode_id"], how="left", validate="many_to_one",
)
elapsed = (
    pd.to_datetime(series_summary["observed_to"], utc=True)
    - pd.to_datetime(series_summary["observed_from"], utc=True)
).dt.total_seconds()
periodic = series_summary["sampling_mode"].eq("periodic")
series_summary["expected_rows"] = np.where(
    periodic,
    np.floor(elapsed / series_summary["expected_cadence_seconds"]) + 1,
    np.nan,
)
series_summary["coverage"] = series_summary["rows"] / series_summary["expected_rows"]
series_summary["series_iqr"] = series_summary["q75"] - series_summary["q25"]
series_summary["constant_series"] = (
    series_summary["rows"].gt(1) & series_summary["minimum"].eq(series_summary["maximum"])
)

quality = series_summary.groupby("metric_id", as_index=False).agg(
    available_episodes=("episode_id", "size"),
    valid_rate_median=("valid_rate", "median"),
    coverage_median=("coverage", "median"),
    clipped_rate_p90=("clipped_rate", lambda x: x.quantile(0.90)),
    constant_series_rate=("constant_series", "mean"),
    median_rows_per_series=("rows", "median"),
    series_median_p10=("series_median", lambda x: x.quantile(0.10)),
    series_median_p50=("series_median", "median"),
    series_median_p90=("series_median", lambda x: x.quantile(0.90)),
    series_iqr_median=("series_iqr", "median"),
)
quality["availability_rate"] = quality["available_episodes"] / len(episodes)

metric_evidence = (
    catalogue.merge(value_summary, on="metric_id", how="left")
    .merge(quality, on="metric_id", how="left")
)
central_span = metric_evidence["q99"] - metric_evidence["q01"]
extreme_span = np.maximum(
    (metric_evidence["maximum"] - metric_evidence["median"]).abs(),
    (metric_evidence["minimum"] - metric_evidence["median"]).abs(),
)
metric_evidence["extreme_to_central_span"] = np.where(
    central_span.gt(0), extreme_span / central_span, np.nan
)
metric_evidence["variance_to_mean"] = np.where(
    metric_evidence["measurement_kind"].eq("interval_count")
    & metric_evidence["mean"].gt(0),
    metric_evidence["variance"] / metric_evidence["mean"],
    np.nan,
)

display(structural)
display(metric_evidence)
assert int(structural.loc[0, "duplicate_keys"]) == 0


## 4. Representative episodes, cadence and missingness

A reproducible hash sample of entities is used for plots. Within each selected
entity, the episode closest to its typical size and metric availability is used.
Cadence is measured
inside each `(entity, episode, metric)` series; differences never cross an
episode boundary.


In [ ]:
episode_sizes = connection.execute('''
    SELECT entity_id, episode_id, count(*) AS rows,
           count(DISTINCT metric_id) AS available_metrics,
           avg(CASE WHEN quality_code <> 'invalid' AND value IS NOT NULL
                    THEN 1.0 ELSE 0.0 END) AS valid_rate
    FROM telemetry GROUP BY entity_id, episode_id
''').df()

def stable_key(value):
    return hashlib.sha256(str(value).encode()).hexdigest()

available_entities = sorted(
    episode_sizes["entity_id"].astype(str).unique(), key=stable_key
)
selected_entities = list(ENTITY_IDS) if ENTITY_IDS else available_entities[:ENTITY_LIMIT]
if set(selected_entities) - set(available_entities):
    raise ValueError("EDA_ENTITY_IDS contains an unknown entity")

candidates = episode_sizes.loc[episode_sizes["entity_id"].astype(str).isin(selected_entities)].copy()
typical_rows = candidates.groupby("entity_id")["rows"].transform("median")
typical_metrics = candidates.groupby("entity_id")["available_metrics"].transform("median")
candidates["selection_distance"] = (
    (candidates["rows"] - typical_rows).abs() / typical_rows.clip(lower=1)
    + (candidates["available_metrics"] - typical_metrics).abs()
      / typical_metrics.clip(lower=1)
)
selected_episodes = (
    candidates.sort_values(["entity_id", "selection_distance", "episode_id"])
    .drop_duplicates("entity_id").reset_index(drop=True)
)
while selected_episodes["rows"].sum() > MAX_ANALYSIS_ROWS and len(selected_episodes) > 1:
    largest = selected_episodes["rows"].idxmax()
    selected_episodes = selected_episodes.drop(index=largest).reset_index(drop=True)
if selected_episodes["rows"].sum() > MAX_ANALYSIS_ROWS:
    raise MemoryError("Choose a smaller episode with EDA_ENTITY_IDS")

connection.register("selected_episodes", selected_episodes[["entity_id", "episode_id"]])
analysis = connection.execute('''
    SELECT t.* FROM telemetry AS t
    JOIN selected_episodes AS s USING (entity_id, episode_id)
    ORDER BY entity_id, episode_id, metric_id, event_ts
''').df()
analysis["event_ts"] = pd.to_datetime(analysis["event_ts"], utc=True)
analysis["value"] = pd.to_numeric(analysis["value"], errors="coerce")

ordered = analysis.sort_values(["entity_id", "episode_id", "metric_id", "event_ts"]).copy()
ordered["delta_seconds"] = ordered.groupby(
    ["entity_id", "episode_id", "metric_id"]
)["event_ts"].diff().dt.total_seconds()
cadence_summary = (
    ordered.dropna(subset=["delta_seconds"])
    .groupby("metric_id", as_index=False)["delta_seconds"]
    .agg(intervals="size", cadence_p10=lambda x: x.quantile(.10),
         observed_cadence="median", cadence_p90=lambda x: x.quantile(.90))
    .merge(catalogue[["metric_id", "expected_cadence_seconds"]], on="metric_id", how="left")
)

gap_columns = [
    "metric_id", "gaps", "affected_entities", "missing_intervals",
    "median_gap_minutes", "p95_gap_minutes", "maximum_gap_minutes",
]
gap_scope_columns = [
    "metrics_affected", "gap_patterns", "affected_entities",
    "median_gap_minutes", "maximum_gap_minutes",
]
gap_summary = pd.DataFrame(columns=gap_columns)
gap_scope_summary = pd.DataFrame(columns=gap_scope_columns)
if not collection_gaps.empty:
    gaps = collection_gaps.copy()
    gaps["gap_minutes"] = (
        pd.to_datetime(gaps["gap_end"], utc=True)
        - pd.to_datetime(gaps["gap_start"], utc=True)
    ).dt.total_seconds() / 60
    gaps["missing_intervals"] = np.ceil(
        gaps["gap_minutes"] * 60 / gaps["expected_cadence_seconds"]
    ).astype("int64")
    gap_summary = gaps.groupby("metric_id", as_index=False).agg(
        gaps=("gap_start", "size"),
        affected_entities=("entity_id", "nunique"),
        missing_intervals=("missing_intervals", "sum"),
        median_gap_minutes=("gap_minutes", "median"),
        p95_gap_minutes=("gap_minutes", lambda x: x.quantile(0.95)),
        maximum_gap_minutes=("gap_minutes", "max"),
    )[gap_columns]

    gap_patterns = gaps.groupby(
        ["entity_id", "episode_id", "gap_start", "gap_end"], as_index=False
    ).agg(metrics_affected=("metric_id", "nunique"))
    gap_patterns["gap_minutes"] = (
        pd.to_datetime(gap_patterns["gap_end"], utc=True)
        - pd.to_datetime(gap_patterns["gap_start"], utc=True)
    ).dt.total_seconds() / 60
    gap_scope_summary = gap_patterns.groupby("metrics_affected", as_index=False).agg(
        gap_patterns=("gap_start", "size"),
        affected_entities=("entity_id", "nunique"),
        median_gap_minutes=("gap_minutes", "median"),
        maximum_gap_minutes=("gap_minutes", "max"),
    )[gap_scope_columns]

display(selected_episodes)
display(cadence_summary)
display(gap_summary)
display(gap_scope_summary)
print(f"Analysis rows in memory: {len(analysis):,}")


In [ ]:
selected_keys = selected_episodes[["entity_id", "episode_id"]].copy()
selected_keys["series_id"] = (
    selected_keys["entity_id"].astype(str) + " | " + selected_keys["episode_id"].astype(str)
)
grid = selected_keys.merge(catalogue[["metric_id"]], how="cross").merge(
    series_summary[["entity_id", "episode_id", "metric_id", "valid_rate", "coverage"]],
    on=["entity_id", "episode_id", "metric_id"], how="left",
)
grid["available"] = grid["valid_rate"].notna()

for column, title in [
    ("available", "Metric available in episode"),
    ("valid_rate", "Valid-value rate"),
    ("coverage", "Periodic observation coverage"),
]:
    if column == "coverage" and grid[column].notna().sum() == 0:
        continue
    matrix = grid.pivot(index="series_id", columns="metric_id", values=column)
    fig, ax = plt.subplots(figsize=(max(10, .55 * len(matrix.columns)), max(3, .55 * len(matrix))))
    sns.heatmap(matrix, vmin=0, vmax=1, cmap="viridis", ax=ax)
    ax.set_title(title)
    save_plot(fig, f"heatmap_{column}.png")


## 5. Distributions and representative time series

Plots respect measurement kind: states use frequencies, counts use a `log1p`
distribution, counters are inspected through increments, and continuous
measurements use observed levels. Missing values are not interpolated, and
plots and rolling summaries stop at collection gaps.


In [ ]:
available = metric_evidence.loc[
    metric_evidence["metric_id"].isin(analysis["metric_id"].unique())
].sort_values(
    ["constant_series_rate", "availability_rate", "rows"],
    ascending=[True, False, False],
)
states = available.loc[available["measurement_kind"].eq("discrete_state")]
non_states = available.loc[available["measurement_kind"].ne("discrete_state")]
state_limit = min(MAX_STATE_PLOTS, len(states), MAX_PLOT_METRICS)
non_state_limit = MAX_PLOT_METRICS - state_limit

plot_metrics = []
for kind in non_states["measurement_kind"].drop_duplicates():
    plot_metrics.append(
        non_states.loc[non_states["measurement_kind"].eq(kind), "metric_id"].iloc[0]
    )
for metric_id in non_states["metric_id"]:
    if metric_id not in plot_metrics and len(plot_metrics) < non_state_limit:
        plot_metrics.append(metric_id)
plot_metrics = plot_metrics[:non_state_limit] + states["metric_id"].head(state_limit).tolist()

catalogue_index = catalogue.set_index("metric_id")
for metric_id in plot_metrics:
    kind = catalogue_index.loc[metric_id, "measurement_kind"]
    values = analysis.loc[
        analysis["metric_id"].eq(metric_id) & analysis["quality_code"].ne("invalid"), "value"
    ].dropna()
    if len(values) > 50_000:
        values = values.sample(50_000, random_state=42)
    if values.empty:
        print(f"{metric_id}: no valid values in selected episodes")
        continue
    if kind == "discrete_state":
        fig, ax = plt.subplots(figsize=(8, 4))
        values.value_counts().sort_index().plot.bar(ax=ax)
        ax.set_title(f"{metric_id}: state frequencies")
    else:
        shown = np.log1p(values.clip(lower=0)) if kind == "interval_count" else values
        lower, upper = shown.quantile([.005, .995])
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(shown.loc[shown.between(lower, upper)], bins=40, ax=axes[0])
        sns.boxplot(x=shown, showfliers=False, ax=axes[1])
        axes[0].set_title(f"{metric_id}: central 99%")
        axes[1].set_title(f"{metric_id}: robust boxplot")
    save_plot(fig, f"distribution_{metric_id}.png")

display(metric_evidence[[
    "metric_id", "measurement_kind", "unit", "rows", "valid_rate",
    "availability_rate", "coverage_median", "clipped_rate", "zero_rate",
    "finite_rate", "minimum", "q01", "q05", "median", "q95",
    "q99", "maximum", "extreme_to_central_span", "variance_to_mean",
]])


In [ ]:
def choose_series(metric_id):
    groups = list(analysis.loc[analysis["metric_id"].eq(metric_id)].groupby(
        ["entity_id", "episode_id"], sort=True
    ))
    lengths = np.array([group["value"].notna().sum() for _, group in groups])
    return groups[np.argmin(abs(lengths - np.median(lengths)))]

def observed_series(frame):
    valid = frame.loc[
        frame["quality_code"].ne("invalid") & frame["value"].notna(), ["event_ts", "value"]
    ].drop_duplicates("event_ts").sort_values("event_ts")
    return valid.set_index("event_ts")["value"]

def contiguous_segments(series, cadence_seconds):
    if series.empty or pd.isna(cadence_seconds):
        return [series]
    cadence = pd.Timedelta(seconds=float(cadence_seconds))
    groups = series.index.to_series().diff().gt(cadence * 1.5).cumsum()
    return [part for _, part in series.groupby(groups)]

def longest_segment(series, cadence_seconds):
    return max(
        contiguous_segments(series, cadence_seconds), key=len, default=series
    )

representatives, temporal_rows = {}, []
for metric_id in plot_metrics:
    identity, frame = choose_series(metric_id)
    metadata = catalogue_index.loc[metric_id]
    kind = metadata["measurement_kind"]
    series = observed_series(frame)
    segment = longest_segment(series, metadata["expected_cadence_seconds"])
    differences = segment.diff().dropna()
    representatives[metric_id] = segment
    temporal_rows.append({
        "metric_id": metric_id, "entity_id": str(identity[0]), "episode_id": str(identity[1]),
        "observations": len(series), "longest_contiguous": len(segment),
        "state_transition_rate": differences.ne(0).mean() if kind == "discrete_state" else np.nan,
        "counter_reset_rate": differences.lt(0).mean() if kind == "cumulative_counter" else np.nan,
        "median_counter_increment": differences.median() if kind == "cumulative_counter" else np.nan,
    })

    fig, ax = plt.subplots(figsize=(12, 4))
    step = max(1, len(series) // 5_000)
    window = min(60, max(5, len(series) // 10))
    for index, part in enumerate(
        contiguous_segments(series, metadata["expected_cadence_seconds"])
    ):
        shown = part.iloc[::step]
        ax.plot(
            shown.index, shown, color="tab:blue", alpha=.55,
            drawstyle="steps-post" if kind == "discrete_state" else "default",
            label="value" if index == 0 else None,
        )
        if kind not in {"discrete_state", "cumulative_counter"} and len(part) >= 5:
            min_periods = min(len(part), max(3, window // 3))
            rolling = part.rolling(window, min_periods=min_periods)
            median_line = rolling.median().iloc[::step]
            q25, q75 = rolling.quantile(.25).iloc[::step], rolling.quantile(.75).iloc[::step]
            ax.plot(
                median_line, color="black",
                label=f"rolling median ({window})" if index == 0 else None,
            )
            ax.fill_between(
                q25.index, q25, q75, color="tab:blue", alpha=.15,
                label="rolling IQR" if index == 0 else None,
            )
    ax.set_title(f"{metric_id} | {identity[0]} | {identity[1]}")
    ax.legend()
    save_plot(fig, f"series_{metric_id}.png")

temporal_evidence = pd.DataFrame(temporal_rows)
display(temporal_evidence)


## 6. Temporal dependence and seasonality

Diagnostics use the longest contiguous observed segment—never interpolated
data. Counts use `log1p`, counters use first differences, and states are
excluded. ADF/KPSS and STL are limited to continuous measurements; their
results are evidence rather than automatic modelling decisions. STL requires
at least six contiguous cycles.


In [ ]:
stationarity_rows, seasonality_rows = [], []
periods = {"daily": 86_400, "weekly": 604_800}

for metric_id, segment in representatives.items():
    metadata = catalogue_index.loc[metric_id]
    kind = metadata["measurement_kind"]
    if kind == "discrete_state":
        continue
    signal = (
        np.log1p(segment.clip(lower=0)) if kind == "interval_count"
        else segment.diff().dropna() if kind == "cumulative_counter"
        else segment
    ).dropna()

    if len(signal) >= 100 and signal.nunique() >= 3:
        fig, ax = plt.subplots(figsize=(9, 4))
        daily_lag = (
            round(86_400 / float(metadata["expected_cadence_seconds"]))
            if metadata["sampling_mode"] == "periodic" else 0
        )
        acf_lags = min(max(60, daily_lag + 12), len(signal) // 4, 200)
        plot_acf(signal.iloc[:10_000], lags=acf_lags, zero=False, ax=ax)
        if 0 < daily_lag <= acf_lags:
            ax.axvline(daily_lag, color="tab:orange", linestyle="--", label="daily lag")
            ax.legend()
        ax.set_title(f"{metric_id}: ACF of diagnostic signal")
        save_plot(fig, f"acf_{metric_id}.png")

    if kind in {"gauge", "bounded_fraction"} and len(signal) >= 100 and signal.nunique() >= 5:
        values = signal.iloc[:10_000]
        result = {"metric_id": metric_id, "observations": len(values),
                  "span_hours": (values.index[-1] - values.index[0]).total_seconds() / 3600,
                  "adf_pvalue": np.nan, "kpss_pvalue": np.nan}
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                result["adf_pvalue"] = adfuller(values, autolag="AIC")[1]
                result["kpss_pvalue"] = kpss(values, regression="c", nlags="auto")[1]
        except (ValueError, np.linalg.LinAlgError):
            pass
        stationarity_rows.append(result)

    if (kind in {"gauge", "bounded_fraction"}
            and metadata["sampling_mode"] == "periodic"
            and pd.notna(metadata["expected_cadence_seconds"])):
        for name, seconds in periods.items():
            period = round(seconds / float(metadata["expected_cadence_seconds"]))
            status = "insufficient_contiguous_cycles"
            trend_strength, seasonal_strength = np.nan, np.nan
            if (period >= 2 and len(segment) >= MIN_SEASONAL_CYCLES * period
                    and segment.nunique() >= 5):
                try:
                    stl = STL(segment.astype(float), period=period, robust=True).fit()
                    figure = stl.plot()
                    figure.set_size_inches(11, 7)
                    figure.suptitle(f"{metric_id}: robust STL ({name})", y=1.01)
                    save_plot(figure, f"stl_{metric_id}_{name}.png")
                    residual_variance = np.nanvar(stl.resid)
                    trend_denominator = np.nanvar(stl.trend + stl.resid)
                    seasonal_denominator = np.nanvar(stl.seasonal + stl.resid)
                    trend_strength = (
                        max(0.0, 1 - residual_variance / trend_denominator)
                        if trend_denominator > 0 else 0.0
                    )
                    seasonal_strength = (
                        max(0.0, 1 - residual_variance / seasonal_denominator)
                        if seasonal_denominator > 0 else 0.0
                    )
                    status = "evaluated"
                except (ValueError, np.linalg.LinAlgError):
                    status = "fit_failed"
            seasonality_rows.append({
                "metric_id": metric_id, "period": name,
                "cycles_available": len(segment) / period, "status": status,
                "trend_strength": trend_strength,
                "seasonal_strength": seasonal_strength,
            })

stationarity_summary = pd.DataFrame(
    stationarity_rows,
    columns=["metric_id", "observations", "span_hours", "adf_pvalue", "kpss_pvalue"],
)
seasonality_summary = pd.DataFrame(
    seasonality_rows,
    columns=[
        "metric_id", "period", "cycles_available", "status",
        "trend_strength", "seasonal_strength",
    ],
)
display(stationarity_summary)
display(seasonality_summary)


## 7. Cross-metric dependence

Only gauges and bounded fractions sharing one cadence are compared. Correlation
is calculated within episodes and summarized across them. Pair-specific episode
and observation counts prevent sparse matrices from appearing well supported.
Counters and states are deliberately excluded from ordinary level correlation.


In [ ]:
continuous = metric_evidence.loc[
    metric_evidence["measurement_kind"].isin(["gauge", "bounded_fraction"])
    & metric_evidence["expected_cadence_seconds"].notna()
].sort_values(["availability_rate", "constant_series_rate"], ascending=[False, True])
pair_rows = []

for cadence, metadata in continuous.groupby("expected_cadence_seconds"):
    metrics = metadata["metric_id"].head(12).tolist()
    if len(metrics) < 2:
        continue
    valid = analysis.loc[
        analysis["metric_id"].isin(metrics)
        & analysis["quality_code"].ne("invalid")
        & analysis["value"].notna()
    ]
    for identity, episode in valid.groupby(
        ["entity_id", "episode_id"]
    ):
        pivot = episode.pivot_table(
            index="event_ts", columns="metric_id", values="value"
        ).reindex(columns=metrics).sort_index()
        if len(pivot) < 30:
            continue
        change = pivot.diff()
        breaks = pivot.index.to_series().diff().dt.total_seconds().gt(float(cadence) * 1.5)
        change.loc[breaks.to_numpy(), :] = np.nan
        for left_index, left in enumerate(metrics):
            for right in metrics[left_index + 1:]:
                level_pairs = pivot[[left, right]].dropna()
                difference_pairs = change[[left, right]].dropna()
                if len(level_pairs) < 30 and len(difference_pairs) < 30:
                    continue
                pair_rows.append({
                    "cadence_seconds": cadence,
                    "entity_id": str(identity[0]),
                    "episode_id": str(identity[1]),
                    "metric_left": left, "metric_right": right,
                    "level_spearman": (
                        level_pairs.corr(method="spearman").iloc[0, 1]
                        if len(level_pairs) >= 30 else np.nan
                    ),
                    "difference_spearman": (
                        difference_pairs.corr(method="spearman").iloc[0, 1]
                        if len(difference_pairs) >= 30 else np.nan
                    ),
                    "level_pairs": len(level_pairs),
                    "difference_pairs": len(difference_pairs),
                })

dependence_columns = [
    "cadence_seconds", "metric_left", "metric_right",
    "level_spearman", "difference_spearman",
    "level_episodes", "difference_episodes",
    "median_level_pairs", "median_difference_pairs",
]
dependence_evidence = pd.DataFrame(columns=dependence_columns)
if pair_rows:
    pair_evidence = pd.DataFrame(pair_rows)
    dependence_evidence = pair_evidence.groupby(
        ["cadence_seconds", "metric_left", "metric_right"], as_index=False
    ).agg(
        level_spearman=("level_spearman", "median"),
        difference_spearman=("difference_spearman", "median"),
        level_episodes=("level_spearman", "count"),
        difference_episodes=("difference_spearman", "count"),
        median_level_pairs=("level_pairs", "median"),
        median_difference_pairs=("difference_pairs", "median"),
    )[dependence_columns]

    for cadence, evidence in dependence_evidence.groupby("cadence_seconds"):
        metrics = sorted(set(evidence["metric_left"]) | set(evidence["metric_right"]))
        level_corr = evidence.pivot(index="metric_left", columns="metric_right", values="level_spearman")
        difference_corr = evidence.pivot(
            index="metric_left", columns="metric_right", values="difference_spearman"
        )
        level_corr = level_corr.reindex(index=metrics, columns=metrics)
        difference_corr = difference_corr.reindex(index=metrics, columns=metrics)
        level_corr = level_corr.combine_first(level_corr.T)
        difference_corr = difference_corr.combine_first(difference_corr.T)
        np.fill_diagonal(level_corr.values, 1.0)
        np.fill_diagonal(difference_corr.values, 1.0)
        fig, axes = plt.subplots(1, 2, figsize=(17, 7))
        sns.heatmap(level_corr, vmin=-1, vmax=1, center=0, cmap="vlag", ax=axes[0])
        sns.heatmap(difference_corr, vmin=-1, vmax=1, center=0, cmap="vlag", ax=axes[1])
        axes[0].set_title("Median within-episode levels")
        axes[1].set_title("Median within-episode first differences")
        save_plot(fig, f"correlation_{int(cadence)}s.png")

display(dependence_evidence.head(20))


## 8. Save compact evidence

The outputs are summary tables and figures, not another copy of telemetry.
Threshold-free evidence is retained for feature engineering and baseline design.


In [ ]:
metric_evidence = metric_evidence.merge(
    cadence_summary[["metric_id", "observed_cadence"]], on="metric_id", how="left"
)
display(metric_evidence[[
    "metric_id", "measurement_kind", "sampling_mode", "expected_cadence_seconds",
    "observed_cadence", "available_episodes", "availability_rate",
    "valid_rate_median", "coverage_median", "clipped_rate_p90",
    "constant_series_rate",
]])

tables = {
    "structural_summary": structural,
    "selected_episodes": selected_episodes,
    "metric_evidence": metric_evidence,
    "series_summary": series_summary,
    "cadence_summary": cadence_summary,
    "gap_summary": gap_summary,
    "gap_scope_summary": gap_scope_summary,
    "temporal_evidence": temporal_evidence,
    "stationarity_summary": stationarity_summary,
    "seasonality_summary": seasonality_summary,
    "dependence_evidence": dependence_evidence,
}
summary = {
    "eda_version": EDA_VERSION,
    "contract_version": CORE_VERSION,
    "sector": SECTOR,
    "canonical_run_id": CANONICAL_RUN_ID,
    "core_fingerprint": core_manifest["fingerprint"],
    "configuration": {
        "entity_limit": ENTITY_LIMIT,
        "max_analysis_rows": MAX_ANALYSIS_ROWS,
        "max_plot_metrics": MAX_PLOT_METRICS,
        "max_state_plots": MAX_STATE_PLOTS,
        "min_seasonal_cycles": MIN_SEASONAL_CYCLES,
        "entity_selection": "stable_sha256_sample",
        "episode_selection": "closest_to_entity_median_rows_and_metric_count",
    },
    "selected_entities": selected_episodes["entity_id"].astype(str).tolist(),
    "selected_episodes": selected_episodes["episode_id"].astype(str).tolist(),
    "truth_guard_passed": True,
    "output_rows": {name: len(frame) for name, frame in tables.items()},
    "figures": sorted(path.name for path in FIGURE_CACHE.glob("*.png")),
}

if SAVE_OUTPUTS:
    with new_output_directory(EDA_ROOT) as output:
        for name, frame in tables.items():
            frame.to_parquet(output / f"{name}.parquet", index=False)
        shutil.copytree(FIGURE_CACHE, output / "figures")
        write_json(output / "eda_summary.json", summary)
    print("Saved EDA evidence:", EDA_ROOT)

display(pd.Series({
    "truth_guard_passed": True,
    "duplicate_keys": int(structural.loc[0, "duplicate_keys"]),
    "metrics_profiled": len(metric_evidence),
    "figures": len(summary["figures"]),
    "next_stage": "Notebook 03 evaluation harness",
}, name="result").to_frame())

connection.close()
figure_temp.cleanup()
